# **Data Collection**

## Objectives

* Download data from Kaggle.com and perform an initial EDA.

## Inputs

* unclean_smartwatch_health_data.csv

## Outputs

* ydata-profiling EDA

## Additional Comments

* In case you have any additional comments that don't fit in the previous bullets, please state them here. 

---

# Change working directory

* We are assuming you will store the notebooks in a subfolder, therefore when running the notebook in the editor, you will need to change the working directory

We need to change the working directory from its current folder to its parent folder
* We access the current directory with os.getcwd()

In [1]:
import os
current_dir = os.getcwd()
current_dir

'/workspaces/Predictive_Analytics_Project/jupyter_notebooks'

We want to make the parent of the current directory the new current directory
* os.path.dirname() gets the parent directory
* os.chir() defines the new current directory

In [2]:
os.chdir(os.path.dirname(current_dir))
print("You set a new current directory")

You set a new current directory


Confirm the new current directory

In [3]:
current_dir = os.getcwd()
current_dir

'/workspaces/Predictive_Analytics_Project'

Setup needed variables

In [4]:
InputFolder = "inputs/"
OutputFolder = "outputs/"
UntouchedData = InputFolder + "smartwatch_health_data_untouched/"
CleanedData = InputFolder + "cleaned_data/"


## Step 1: Load Data

Load data and drop User ID

In [5]:
import pandas as pd

df = pd.read_csv(UntouchedData + "unclean_smartwatch_health_data.csv").drop("User ID", axis=1)
df.head()

df_test = pd.read_csv(CleanedData + "smoothed_data/smoothed_smartwatch_health_data.csv")
df_test.value_counts()

Heart Rate (BPM)  Blood Oxygen Level (%)  Step Count   Sleep Duration (hours)  Stress Level  Activity Level
91.364690         99.205466               311.301541   6.347072                6.000000      Highly Active     3
61.142563         97.706217               5704.336852  7.710167                9.000000      Highly Active     3
80.787307         99.068231               2091.048179  6.861518                4.666667      Highly Active     3
64.204102         97.034262               5510.503118  5.905802                4.666667      Sedentary         3
57.334290         99.264884               4113.228546  6.624580                6.666667      Highly Active     3
                                                                                                              ..
70.066166         98.201064               725.472727   5.762139                6.000000      Highly Active     1
70.074271         96.227274               9328.969516  6.293065                6.000000      Active  

## Step 2: Data Cleaning

Declare custom transformers

In [36]:
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin

# Replace 'Very High' with 10 in 'Stress Level' and convert columns to numeric
class DataTypeTransformer(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        # Replace 'Very High' with 10 in 'Stress Level'
        X['Stress Level'] = X['Stress Level'].replace('Very High', 10)

        # Convert columns to numeric, handling non-numeric values
        X['Sleep Duration (hours)'] = pd.to_numeric(X['Sleep Duration (hours)'], errors='coerce')
        X['Stress Level'] = pd.to_numeric(X['Stress Level'], errors='coerce')

        return X


# Impute missing values (not for Activity Level)
from sklearn.impute import KNNImputer

class KNNImputerTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, n_neighbors=3):
        self.n_neighbors = n_neighbors
        self.imputer = KNNImputer(n_neighbors=self.n_neighbors)

    def fit(self, X, y=None):
        # Identify numerical columns
        self.num_cols = X.select_dtypes(include=[np.number]).columns
        self.imputer.fit(X[self.num_cols])
        return self

    def transform(self, X):
        X = X.copy()
        # Impute numerical columns
        X[self.num_cols] = self.imputer.transform(X[self.num_cols])
        return X


# Handle Outliers
from feature_engine.outliers import Winsorizer

class WinsorizerTransformer(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.winsorizers = {}

    def fit(self, X, y=None):
        X = X.copy()
        # Define and fit Winsorizers for each variable
        self.winsorizers['Sleep Duration (hours)'] = Winsorizer(
            capping_method='iqr', tail='both', fold=1.5, variables=['Sleep Duration (hours)']
        ).fit(X)
        self.winsorizers['Heart Rate (BPM)'] = Winsorizer(
            capping_method='iqr', tail='right', fold=1.5, variables=['Heart Rate (BPM)']
        ).fit(X)
        self.winsorizers['Blood Oxygen Level (%)'] = Winsorizer(
            capping_method='iqr', tail='left', fold=1.5, variables=['Blood Oxygen Level (%)']
        ).fit(X)
        self.winsorizers['Step Count'] = Winsorizer(
            capping_method='iqr', tail='right', fold=1.5, variables=['Step Count']
        ).fit(X)
        return self

    def transform(self, X):
        X = X.copy()
        # Apply Winsorizers
        for winsorizer in self.winsorizers.values():
            X = winsorizer.transform(X)
        return X


class FloatToInt(BaseEstimator, TransformerMixin):
    def __init__(self, columns=['Stress Level']):
        self.columns = columns

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        # Convert specified columns to integer
        for col in self.columns:
            X[col] = X[col].astype(int)
        return X

# Predict missing values in 'Activity Level'
from sklearn.ensemble import RandomForestClassifier

class CategoricalImputer(BaseEstimator, TransformerMixin):
    def __init__(self, target_column='Activity Level', random_state=42):
        self.target_column = target_column
        self.random_state = random_state
        self.classifier = RandomForestClassifier(random_state=self.random_state)
        self.original_categories = None

    def fit(self, X, y=None):
        X = X.copy()
        # Save original categories
        X[self.target_column] = X[self.target_column].astype('category')
        self.original_categories = X[self.target_column].cat.categories

        # Encode target column
        X[self.target_column] = X[self.target_column].cat.codes.replace(-1, np.nan)

        # Split data
        self.train_data = X.dropna(subset=[self.target_column])
        self.test_data = X[X[self.target_column].isna()]

        # Features and target
        self.X_train = self.train_data.drop(columns=[self.target_column])
        self.y_train = self.train_data[self.target_column].astype(int)

        # Fit classifier
        self.classifier.fit(self.X_train, self.y_train)
        return self

    def transform(self, X):
        X = X.copy()
        # Encode target column
        X[self.target_column] = X[self.target_column].astype('category')
        X[self.target_column] = X[self.target_column].cat.codes.replace(-1, np.nan)

        # Identify missing values
        missing_mask = X[self.target_column].isna()
        if missing_mask.any():
            X_missing = X[missing_mask]
            X_missing_features = X_missing.drop(columns=[self.target_column])
            # Predict missing values
            X.loc[missing_mask, self.target_column] = self.classifier.predict(X_missing_features)

        # Convert to int and decode categories
        X[self.target_column] = X[self.target_column].astype(int)
        X[self.target_column] = pd.Categorical.from_codes(
            X[self.target_column], categories=self.original_categories
        )
        return X


# Correct mis-spelled values in Activity Level
class CategoryCorrector(BaseEstimator, TransformerMixin):
    def __init__(self, column='Activity Level'):
        self.column = column
        self.class_mapping = {
            'Seddentary': 'Sedentary',
            'Highly_Active': 'Highly Active',
            'Actve': 'Active'
        }

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        X[self.column] = X[self.column].replace(self.class_mapping)
        return X


# Smooth data using K-Nearest Neighbors
from sklearn.neighbors import NearestNeighbors

class DataSmoother(BaseEstimator, TransformerMixin):
    def __init__(self, k=3):
        self.k = k
        self.nn = NearestNeighbors(n_neighbors=self.k)

    def fit(self, X, y=None):
        X = X.copy()
        self.num_cols = X.select_dtypes(include=[np.number]).columns
        self.nn.fit(X[self.num_cols])
        return self

    def transform(self, X):
        X = X.copy()
        distances, indices = self.nn.kneighbors(X[self.num_cols])

        # Smooth numerical columns
        for i, col in enumerate(self.num_cols):
            X[col] = [np.mean(X.iloc[indices[row_idx]][col]) for row_idx in range(len(X))]
        return X


# Trim outliers again after smoothing
from feature_engine.outliers import OutlierTrimmer

class OutlierTrimmerTransformer(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.trimmers = {}

    def fit(self, X, y=None):
        X = X.copy()
        # Define and fit Outlier Trimmers for each variable
        self.trimmers['Heart Rate (BPM)'] = OutlierTrimmer(
            capping_method='quantiles', tail='right', fold=0.05, variables=['Heart Rate (BPM)']
        ).fit(X)
        self.trimmers['Blood Oxygen Level (%)'] = OutlierTrimmer(
            capping_method='quantiles', tail='left', fold=0.05, variables=['Blood Oxygen Level (%)']
        ).fit(X)
        self.trimmers['Sleep Duration (hours)'] = OutlierTrimmer(
            capping_method='quantiles', tail='both', fold=0.05, variables=['Sleep Duration (hours)']
        ).fit(X)
        self.trimmers['Step Count'] = OutlierTrimmer(
            capping_method='quantiles', tail='right', fold=0.05, variables=['Step Count']
        ).fit(X)
        return self

    def transform(self, X):
        X = X.copy()
        # Apply Outlier Trimmers sequentially
        for trimmer in self.trimmers.values():
            X = trimmer.transform(X)
        return X

from sklearn.preprocessing import MinMaxScaler
class DataFrameScaler(BaseEstimator, TransformerMixin):
    def __init__(self, exclude_columns=None):
        self.exclude_columns = exclude_columns
        self.scaler = MinMaxScaler()
    
    def fit(self, X, y=None):
        X_to_scale = X.drop(columns=self.exclude_columns)
        self.scaler.fit(X_to_scale)
        return self
    
    def transform(self, X):
        X_to_scale = X.drop(columns=self.exclude_columns)
        X_excluded = X[self.exclude_columns]
        X_scaled = pd.DataFrame(self.scaler.transform(X_to_scale), columns=X_to_scale.columns)
        X_final = pd.concat([X_scaled, X_excluded.reset_index(drop=True)], axis=1)
        return X_final

### Assemble cleaning pipeline

In [27]:
from sklearn.pipeline import Pipeline

# Create the modified data cleaning pipeline
data_cleaning_pipeline = Pipeline([
    ('data_type_transformer', DataTypeTransformer()),
    ('knn_imputer', KNNImputerTransformer(n_neighbors=3)),
    ('winsorizer_transformer', WinsorizerTransformer()),
    ('categorical_imputer', CategoricalImputer()),
    ('category_corrector', CategoryCorrector()),
    ('float_to_int', FloatToInt()),
    ('data_smoother', DataSmoother(k=3)),
    ('outlier_trimmer_transformer', OutlierTrimmerTransformer())
])


In [28]:
df_cleaned = data_cleaning_pipeline.fit_transform(df)

/tmp/ipykernel_126911/3950964743.py:159: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  X[self.column] = X[self.column].replace(self.class_mapping)


In [25]:
# Apply each transformer individually and inspect the result
intermediate_data = DataTypeTransformer().fit_transform(df.copy())
print("After DataTypeTransformer:")
print(intermediate_data['Blood Oxygen Level (%)'])

intermediate_data = KNNImputerTransformer(n_neighbors=3).fit_transform(intermediate_data)
print("After KNNImputerTransformer:")
print(intermediate_data['Blood Oxygen Level (%)'])

intermediate_data = FloatToInt().fit_transform(intermediate_data)
print("After FloatToInt:")
print(intermediate_data['Blood Oxygen Level (%)'])

intermediate_data = WinsorizerTransformer().fit_transform(intermediate_data)
print("After WinsorizerTransformer:")
print(intermediate_data['Blood Oxygen Level (%)'])

intermediate_data = CategoryCorrector().fit_transform(intermediate_data)
print("After CategoryCorrector:")
print(intermediate_data['Blood Oxygen Level (%)'])

intermediate_data = DataSmoother(k=3).fit_transform(intermediate_data)
print("After DataSmoother:")
print(intermediate_data['Blood Oxygen Level (%)'])

intermediate_data = OutlierTrimmerTransformer().fit_transform(intermediate_data)
print("After OutlierTrimmerTransformer:")
print(intermediate_data['Blood Oxygen Level (%)'])

After DataTypeTransformer:
0       98.809650
1       98.532195
2       97.052954
3       96.894213
4       98.583797
          ...    
9995    98.931927
9996    95.773035
9997    97.945874
9998    98.401058
9999    98.475606
Name: Blood Oxygen Level (%), Length: 10000, dtype: float64
After KNNImputerTransformer:
0       98.809650
1       98.532195
2       97.052954
3       96.894213
4       98.583797
          ...    
9995    98.931927
9996    95.773035
9997    97.945874
9998    98.401058
9999    98.475606
Name: Blood Oxygen Level (%), Length: 10000, dtype: float64
After FloatToInt:
0       98.809650
1       98.532195
2       97.052954
3       96.894213
4       98.583797
          ...    
9995    98.931927
9996    95.773035
9997    97.945874
9998    98.401058
9999    98.475606
Name: Blood Oxygen Level (%), Length: 10000, dtype: float64
After WinsorizerTransformer:
0       98.809650
1       98.532195
2       97.052954
3       96.894213
4       98.583797
          ...    
9995    98.9319

## Step 2: Feature Engineering

We researched for the optimal values for n_components and n_clusters in the engineering notebook

In [40]:
# Normality improvement
from feature_engine.transformation import BoxCoxTransformer
from sklearn.pipeline import Pipeline
# Scaler
from sklearn.preprocessing import MinMaxScaler, StandardScaler
# encoding
from feature_engine.encoding import OrdinalEncoder 
# pca
from sklearn.decomposition import PCA
# kmeans
from sklearn.cluster import KMeans

def ClusteringPipeline():
    cleaning_engineering_pipeline = Pipeline([
        ('data_type_transformer', DataTypeTransformer()),
        ('knn_imputer', KNNImputerTransformer(n_neighbors=3)),
        ('winsorizer_transformer', WinsorizerTransformer()),
        ('categorical_imputer', CategoricalImputer()),
        ('category_corrector', CategoryCorrector()),
        ('float_to_int', FloatToInt()),
        ('data_smoother', DataSmoother(k=3)),
        ('outlier_trimmer_transformer', OutlierTrimmerTransformer()),
        # ('boxcox', BoxCoxTransformer()),
        ('encoder', OrdinalEncoder(encoding_method='arbitrary', variables=["Activity Level"])),
        # I have chosen OrdinalEncoder as our categorical variables have an ordinal relationship
        ('minmax', DataFrameScaler(exclude_columns=["Activity Level"])),
        ('pca', PCA(n_components=3, random_state=42)),
        ('model', KMeans(n_clusters=4, random_state=42))
    ])
    return cleaning_engineering_pipeline

In [31]:
# Apply each transformer individually and inspect the result
intermediate_data = df_cleaned.copy()
print("Original Data:")
print(intermediate_data['Blood Oxygen Level (%)'])

# Apply BoxCoxTransformer
intermediate_data = BoxCoxTransformer().fit_transform(intermediate_data)
print("After BoxCoxTransformer:")
print(intermediate_data['Blood Oxygen Level (%)'])

# Apply OrdinalEncoder
intermediate_data = OrdinalEncoder(encoding_method='arbitrary', variables=["Activity Level"]).fit_transform(intermediate_data)
print("After OrdinalEncoder:")
print(intermediate_data['Blood Oxygen Level (%)'])

# Apply MinMaxScaler
intermediate_data = MinMaxScaler().fit_transform(intermediate_data)
intermediate_data = pd.DataFrame(intermediate_data, columns=df.columns)
print("After MinMaxScaler:")
print(intermediate_data['Blood Oxygen Level (%)'])

# Apply PCA
intermediate_data = PCA(n_components=3, random_state=42).fit_transform(intermediate_data)
intermediate_data = pd.DataFrame(intermediate_data, columns=[f'PCA{i+1}' for i in range(3)])
print("After PCA:")
print(intermediate_data.head())

# Apply KMeans
kmeans = KMeans(n_clusters=5, random_state=42).fit(intermediate_data)
intermediate_data['Clusters'] = kmeans.labels_
print("After KMeans:")
print(intermediate_data.head())

Original Data:
0       97.828229
1       98.201064
3       98.117870
4       97.971623
5       97.092443
          ...    
9995    97.741336
9996    98.227541
9997    98.626366
9998    99.467019
9999    97.847380
Name: Blood Oxygen Level (%), Length: 7797, dtype: float64
After BoxCoxTransformer:
0       647682.025254
1       655542.795044
3       653783.125413
4       650697.639828
5       632358.667756
            ...      
9995    645859.303353
9996    656103.507643
9997    664589.265088
9998    682721.280188
9999    648084.216352
Name: Blood Oxygen Level (%), Length: 7797, dtype: float64
After OrdinalEncoder:
0       647682.025254
1       655542.795044
3       653783.125413
4       650697.639828
5       632358.667756
            ...      
9995    645859.303353
9996    656103.507643
9997    664589.265088
9998    682721.280188
9999    648084.216352
Name: Blood Oxygen Level (%), Length: 7797, dtype: float64
After MinMaxScaler:
0       0.439542
1       0.533864
2       0.512749
3       

/home/cistudent/.local/lib/python3.12/site-packages/sklearn/cluster/_kmeans.py:1412: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)


In [41]:
pipeline_cluster = ClusteringPipeline()
pipeline_pca = Pipeline(pipeline_cluster.steps[:-2])
df_pca = pipeline_pca.fit_transform(df)

print(df_pca.shape,'\n', type(df_pca))

/tmp/ipykernel_126911/1780954305.py:159: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  X[self.column] = X[self.column].replace(self.class_mapping)


(7797, 6) 
 <class 'pandas.core.frame.DataFrame'>


In [ ]:
dfplz = pd.DataFrame(df_pca, columns=df.columns)
dfplz["Blood Oxygen Level (%)"].value_counts()

In [45]:
n_components = 3

pca = PCA(n_components=n_components).fit(df_pca) 
x_PCA = pca.transform(df_pca) 

ComponentsList = ["Component " + str(number) for number in range(n_components)]
dfExplVarRatio = pd.DataFrame(
    data= np.round(100 * pca.explained_variance_ratio_ ,3),
    index=ComponentsList,
    columns=['Explained Variance Ratio (%)'])

# prints how much of the dataset these components explain (naturally in this case will be 100%)
PercentageOfDataExplained = dfExplVarRatio['Explained Variance Ratio (%)'].sum()

print(f"* The {n_components} components explain {round(PercentageOfDataExplained,2)}% of the data \n")
print(dfExplVarRatio)

* The 3 components explain 84.83% of the data 

             Explained Variance Ratio (%)
Component 0                        72.094
Component 1                         6.415
Component 2                         6.320


## Train

In [ ]:
x = df.copy()
pipeline_cluster = ClusteringPipeline()
pipeline_cluster.fit(x)

In [ ]:
import matplotlib.pyplot as plt

# Step 1: Apply the pipeline to transform the data
X = pipeline_cluster.fit_transform(df.copy())

# Convert X to a DataFrame
X = pd.DataFrame(X, columns=['PCA1', 'PCA2'])

# Add the cluster labels
X['Clusters'] = pipeline_cluster['model'].labels_
print(X.shape)
X.head(3)


In [ ]:
import matplotlib.pyplot as plt

print(f"* Clusters frequencies \n{ train['Clusters'].value_counts(normalize=True).to_frame().round(2)} \n\n")
train['Clusters'].value_counts().sort_values().plot(kind='bar')
plt.show()

In [ ]:
import seaborn as sns
sns.set_style("whitegrid")
plt.figure(figsize=(10, 6))
sns.scatterplot(x=train[:, 0], y=train[:, 1],
                hue=X['Clusters'], palette='Set1', alpha=0.6)
plt.scatter(x=pipeline_full['model'].cluster_centers_[:, 0], y=pipeline_full['model'].cluster_centers_[:, 1],
            marker="x", s=169, linewidths=3, color="black")
plt.xlabel("PCA Component 0")
plt.ylabel("PCA Component 1")
plt.title("PCA Components colored by Clusters")
plt.show()

# Section 1

Section 1 content

---

# Section 2 EDA

Section 2 content

---

NOTE

* You may add as many sections as you want, as long as it supports your project workflow.
* All notebook's cells should be run top-down (you can't create a dynamic wherein a given point you need to go back to a previous cell to execute some task, like go back to a previous cell and refresh a variable content)

---

# Push files to Repo

* In case you don't need to push files to Repo, you may replace this section with "Conclusions and Next Steps" and state your conclusions and next steps.

In [ ]:
import os
try:
  # create here your folder
  # os.makedirs(name='')
except Exception as e:
  print(e)
